In [1]:
!pip install streamlit==1.39.0
!pip install Pillow

  Using cached streamlit-1.39.0-py2.py3-none-any.whl.metadata (8.5 kB)
Using cached streamlit-1.39.0-py2.py3-none-any.whl (8.7 MB)
  Attempting uninstall: streamlit
    Found existing installation: streamlit 1.43.2
    Uninstalling streamlit-1.43.2:
      Successfully uninstalled streamlit-1.43.2


In [2]:
pip install --upgrade streamlit


  Using cached streamlit-1.43.2-py2.py3-none-any.whl.metadata (8.9 kB)
Using cached streamlit-1.43.2-py2.py3-none-any.whl (9.7 MB)
  Attempting uninstall: streamlit
    Found existing installation: streamlit 1.39.0
    Uninstalling streamlit-1.39.0:
      Successfully uninstalled streamlit-1.39.0


In [3]:
!pip install SpeechRecognition

In [4]:
!pip install scikit-learn==1.5.2

In [5]:
!pip install pyaudio


In [6]:
!pip install sounddevice


In [7]:
!pip install sounddevice numpy scipy SpeechRecognition

In [1]:
%%writefile app.py
import streamlit as st
import joblib
import nltk
import time
import speech_recognition as sr
from googletrans import Translator
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


# Initialize translator and lemmatizer
translator = Translator()
lemmatizer = WordNetLemmatizer()

# Load the pretrained model and artifacts
MODEL_PATH = 'models/best_random_forest_model_Key_Ex.p'
VECTOR_PATH = 'models/tfidf_vectorizer_Key_Ex.p'
ENCODER_PATH = 'models/label_encoder_Key_Ex.p'

# Load vectorizer and label encoder
vectorizer = joblib.load(VECTOR_PATH)
label_encoder = joblib.load(ENCODER_PATH)

# Load the model
def load_model(model_path):
    """Safely load the model with compatibility handling."""
    try:
        model = joblib.load(model_path)
        return model
    except Exception as e:
        st.error(f"Error loading the model: {e}")
        return None

model = load_model(MODEL_PATH)
if model is None:
    st.error("The model could not be loaded. Please ensure compatibility and file integrity.")
    st.stop()

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Updated predict function
# Emergency keywords for rule-based filtering
emergency_keywords = {
    'Accident_Call': {
        'accident', 'crash', 'collision', 'hit', 'strike', 'vehicle', 'wreck',
        'overturn', 'injured', 'bump', 'pileup', 'unconscious', 'motorcycle',
        'car crash', 'road accident', 'bike accident', 'truck collision', 'pedestrian hit',
        'blood', 'road', 'highway', 'driver', 'fatal', 'ambulance', 'ran over', 'traffic', 'damage',
        'victim', 'impact', 'brakes', 'tyre', 'rash driving'
    },
    'Fire_Call': {
        'fire', 'burn', 'flame', 'blaze', 'ignite', 'smoke', 'cylinder', 'explosion',
        'wildfire', 'combustion', 'scorch', 'ash', 'inferno', 'char', 'sparks', 'extinguish', 'alarm',
        'gas leak', 'short circuit', 'electric fire', 'blast', 'firefighter', 'fire brigade',
        'fumes', 'heat', 'boiler', 'chimney', 'toxic smoke', 'electrical fault', 'flammable', 'fire alarm'
    },
    'Medical_Call': {
        'kidney', 'chest', 'abdomen', 'liver', 'heart', 'lung', 'brain', 'stomach', 'appendix',
        'injury', 'unconscious', 'pain', 'bleed', 'medical', 'sick', 'hurt', 'hurting','faint', 'suicide', 'patient','blood',
        'seizure', 'heartattack', 'breathing', 'allergy', 'stroke', 'vomit','vomiting', 'fever', 'diabetic', 'headache',
        'medicine', 'unwell', 'dizzy', 'infection', 'fracture', 'burns', 'bleeding', 'diabetes', 'epilepsy', 'pulse',
        'intensive care', 'ICU', 'coma', 'emergency room', 'nausea', 'bandage'
    }
}

all_emergency_keywords = set(
    word.lower() for group in emergency_keywords.values() for word in group
)

def predict_emergency(text):
    """Predict the level and type of emergency with rule-based filter."""
    try:
        tokens = word_tokenize(text)
        lemmatized_words = [lemmatizer.lemmatize(w.lower()) for w in tokens if w.isalpha()]
        if not any(word in all_emergency_keywords for word in lemmatized_words):
            return "Normal_Call"
        else:
            transformed_text = vectorizer.transform([text])
            prediction = model.predict(transformed_text)
            predicted_label = label_encoder.inverse_transform(prediction)[0]
            return predicted_label
    except Exception as e:
        return "Normal_Call"


def audio_to_urdu_text(audio_file):
    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(audio_file) as source:
            audio_data = recognizer.record(source)
            return recognizer.recognize_google(audio_data, language='ur')
    except sr.UnknownValueError:
        return "Could not understand the audio."
    except sr.RequestError:
        return "Could not request results from the service."
    except Exception as e:
        return f"Error processing file: {e}"

def record_audio():
    """Record audio from the microphone."""
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        st.info("Recording... Please speak clearly in Urdu.")
        try:
            audio_data = recognizer.listen(source, timeout=30, phrase_time_limit=30)  
            st.success("Recording complete. Processing...")
            return recognizer.recognize_google(audio_data, language='ur')
        except sr.UnknownValueError:
            return "Could not understand the audio."
        except sr.RequestError:
            return "Could not request results from the service."
        except Exception as e:
            return f"An error occurred: {e}"

def show_emergency_response_plan(prediction):
    """Display emergency response plan based on the predicted label."""
    # ➔ Emergency Response Plan (Newly Added)
    emergency_response = {
        "Medical_Call": {
            "workers": 3,
            "vehicles": ["Ambulance"],
            "equipment": ["First Aid Kit", "Oxygen Cylinder", "Stretcher"],
            "instructions": "Dispatch medical team immediately to the location."
        },
        "Fire_Call": {
            "workers": 6,
            "vehicles": ["Fire Truck", "Rescue Vehicle"],
            "equipment": ["Fire Extinguishers", "Water Hoses", "Protective Gear"],
            "instructions": "Dispatch firefighting units and ensure evacuation."
        },
        "Accident_Call": {
            "workers": 4,
            "vehicles": ["Ambulance", "Rescue Vehicle"],
            "equipment": ["First Aid Kit", "Hydraulic Cutters", "Stretchers"],
            "instructions": "Send rescue and medical teams for accident handling."
        },
        "Normal_Call": {
            "workers": 0,
            "vehicles": [],
            "equipment": [],
            "instructions": "No action needed. Monitor if required."
        }
    }

    response_plan = emergency_response.get(prediction, None)
    if response_plan:
        st.subheader("📋 Emergency Response Plan")
        st.markdown(f"""
        *👷 Workers Needed:* {response_plan['workers']}  
        *🚒 Vehicles:* {", ".join(response_plan['vehicles']) if response_plan['vehicles'] else 'None'}  
        *🛠️ Equipment:* {", ".join(response_plan['equipment']) if response_plan['equipment'] else 'None'}  
        *📢 Instructions:* {response_plan['instructions']}
        """)

# Streamlit App
st.set_page_config(page_title="1122 Emergency Detection", page_icon=":ambulance:", layout="wide")
st.title("🚨1122 Emergency Detection System")
st.subheader("AI-powered emergency call detection system")

st.markdown(""" 
This app processes Urdu audio calls to detect emergencies for Pakistan's 1122 emergency response system. 
You can either upload an Urdu audio file or record using the microphone, and the system will analyze it to determine whether it indicates an emergency.
""")

# Sidebar option for selecting input type
input_type = st.sidebar.selectbox("Select Input Type", ["Microphone Input", "Audio File Upload"])

if input_type == "Microphone Input":
    st.markdown("Click the button below to record your voice in Urdu.")
    if st.button("Start Recording"):
        with st.spinner("Recording and processing the audio..."):
            urdu_text = record_audio()
            time.sleep(1)
        
        st.subheader("Audio Conversion")
        col1, col2 = st.columns(2)
        col1.text_area("Converted Urdu Text", urdu_text, height=150)
        
        translated_text = translator.translate(urdu_text, src='ur', dest='en').text if urdu_text else ""
        col2.text_area("Translated English Text", translated_text, height=150)
        
        if translated_text and "Error" not in translated_text:
            with st.spinner("Analyzing the text for emergencies..."):
                prediction = predict_emergency(translated_text)
                time.sleep(1)

            st.subheader("Prediction Result")
            if 'Normal_Call' in prediction:
                st.markdown(
                    f"<div style='padding: 20px; border-radius: 10px; background-color: green; color: white;font-size: 25px; width: 60%;'>"
                    f"<strong>No Emergency:</strong> call is normal."
                    f"</div>", unsafe_allow_html=True
                )
            else:
                st.markdown(
                    f"<div style='padding: 20px; border-radius: 10px; background-color: red; color: white;font-size: 25px; width: 60%;'>"
                    f"<strong>Emergency Detected:</strong> {prediction}"
                    f"</div>", unsafe_allow_html=True
                )

            # Show Emergency Response Plan
            show_emergency_response_plan(prediction)

elif input_type == "Audio File Upload":
    st.sidebar.title("Upload Urdu Audio File")
    audio_file = st.sidebar.file_uploader("Choose an Urdu audio file", type=["wav", "mp3"])
    
    if audio_file is not None:
        with st.spinner("Converting Urdu audio to text..."):
            urdu_text = audio_to_urdu_text(audio_file)
            time.sleep(1)
        
        st.subheader("Audio Conversion")
        col1, col2 = st.columns(2)
        col1.text_area("Converted Urdu Text", urdu_text, height=150)
        
        translated_text = translator.translate(urdu_text, src='ur', dest='en').text if urdu_text else ""
        col2.text_area("Translated English Text", translated_text, height=150)
        
        if translated_text and "Error" not in translated_text:
            with st.spinner("Analyzing the text for emergencies..."):
                prediction = predict_emergency(translated_text)
                time.sleep(1)

            st.subheader("Prediction Result")
            if 'Normal_Call' in prediction:
                st.markdown("<div style='padding: 20px; border-radius: 10px; background-color: green; color: white;font-size: 25px; width: 60%;'><strong>No Emergency:</strong> call is normal.</div>", unsafe_allow_html=True)
            else:
                st.markdown(f"<div style='padding: 20px; border-radius: 10px; background-color: red; color: white;font-size: 25px; width: 60%;'><strong>Emergency Detected:</strong> {prediction}</div>", unsafe_allow_html=True)

            # Show Emergency Response Plan
            show_emergency_response_plan(prediction)

st.markdown("---")
st.markdown("Developed with ❤️ for Pakistan's emergency response system.")

Overwriting app.py


In [ ]:
!streamlit run app.py

In [3]:
%%writefile test.py
import streamlit as st
import joblib
import nltk
import time
import speech_recognition as sr
from googletrans import Translator
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


# Initialize translator and lemmatizer
translator = Translator()
lemmatizer = WordNetLemmatizer()

# Load the pretrained model and artifacts
MODEL_PATH = 'models/best_random_forest_model_Key_Ex.p'
VECTOR_PATH = 'models/tfidf_vectorizer_Key_Ex.p'
ENCODER_PATH = 'models/label_encoder_Key_Ex.p'

# Load vectorizer and label encoder
vectorizer = joblib.load(VECTOR_PATH)
label_encoder = joblib.load(ENCODER_PATH)

# Load the model
def load_model(model_path):
    """Safely load the model with compatibility handling."""
    try:
        model = joblib.load(model_path)
        return model
    except Exception as e:
        st.error(f"Error loading the model: {e}")
        return None

model = load_model(MODEL_PATH)
if model is None:
    st.error("The model could not be loaded. Please ensure compatibility and file integrity.")
    st.stop()

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

# Updated predict function
# Emergency keywords for rule-based filtering
emergency_keywords = {
    'Accident_Call': {
        'accident', 'crash', 'collision', 'hit', 'strike', 'vehicle', 'wreck',
        'overturn', 'injured', 'bump', 'pileup', 'unconscious', 'motorcycle',
        'car crash', 'road accident', 'bike accident', 'truck collision', 'pedestrian hit',
        'blood', 'road', 'highway', 'driver', 'fatal', 'ambulance', 'ran over', 'traffic', 'damage',
        'victim', 'impact', 'brakes', 'tyre', 'rash driving'
    },
    'Fire_Call': {
        'fire', 'burn', 'flame', 'blaze', 'ignite', 'smoke', 'cylinder', 'explosion',
        'wildfire', 'combustion', 'scorch', 'ash', 'inferno', 'char', 'sparks', 'extinguish', 'alarm',
        'gas leak', 'short circuit', 'electric fire', 'blast', 'firefighter', 'fire brigade',
        'fumes', 'heat', 'boiler', 'chimney', 'toxic smoke', 'electrical fault', 'flammable', 'fire alarm'
    },
    'Medical_Call': {
        'kidney', 'chest', 'abdomen', 'liver', 'heart', 'lung', 'brain', 'stomach', 'appendix',
        'injury', 'unconscious', 'pain', 'bleed', 'medical', 'sick', 'hurt', 'hurting','faint', 'suicide', 'patient','blood',
        'seizure', 'heartattack', 'breathing', 'allergy', 'stroke', 'vomit','vomiting', 'fever', 'diabetic', 'headache',
        'medicine', 'unwell', 'dizzy', 'infection', 'fracture', 'burns', 'bleeding', 'diabetes', 'epilepsy', 'pulse',
        'intensive care', 'ICU', 'coma', 'emergency room', 'nausea', 'bandage'
    }
}

all_emergency_keywords = set(
    word.lower() for group in emergency_keywords.values() for word in group
)

def predict_emergency(text):
    """Predict the level and type of emergency with rule-based filter."""
    try:
        tokens = word_tokenize(text)
        lemmatized_words = [lemmatizer.lemmatize(w.lower()) for w in tokens if w.isalpha()]
        if not any(word in all_emergency_keywords for word in lemmatized_words):
            return "Normal_Call"
        else:
            transformed_text = vectorizer.transform([text])
            prediction = model.predict(transformed_text)
            predicted_label = label_encoder.inverse_transform(prediction)[0]
            return predicted_label
    except Exception as e:
        return "Normal_Call"



# Audio file to Urdu text
def audio_to_urdu_text(audio_file):
    """Convert uploaded audio file to Urdu text."""
    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(audio_file) as source:
            audio_data = recognizer.record(source)
            return recognizer.recognize_google(audio_data, language='ur')
    except sr.UnknownValueError:
        return "Could not understand the audio."
    except sr.RequestError:
        return "Could not request results from the service."
    except Exception as e:
        return f"Error processing file: {e}"


def show_emergency_response_plan(prediction):
    """Display emergency response plan based on the predicted label."""
    # ➔ Emergency Response Plan (Newly Added)
    emergency_response = {
        "Medical_Call": {
            "workers": 3,
            "vehicles": ["Ambulance"],
            "equipment": ["First Aid Kit", "Oxygen Cylinder", "Stretcher"],
            "instructions": "Dispatch medical team immediately to the location."
        },
        "Fire_Call": {
            "workers": 6,
            "vehicles": ["Fire Truck", "Rescue Vehicle"],
            "equipment": ["Fire Extinguishers", "Water Hoses", "Protective Gear"],
            "instructions": "Dispatch firefighting units and ensure evacuation."
        },
        "Accident_Call": {
            "workers": 4,
            "vehicles": ["Ambulance", "Rescue Vehicle"],
            "equipment": ["First Aid Kit", "Hydraulic Cutters", "Stretchers"],
            "instructions": "Send rescue and medical teams for accident handling."
        },
        "Normal_Call": {
            "workers": 0,
            "vehicles": [],
            "equipment": [],
            "instructions": "No action needed. Monitor if required."
        }
    }

    response_plan = emergency_response.get(prediction, None)
    if response_plan:
        st.subheader("📋 Emergency Response Plan")
        st.markdown(f"""
        **👷 Workers Needed:** {response_plan['workers']}  
        **🚒 Vehicles:** {", ".join(response_plan['vehicles']) if response_plan['vehicles'] else 'None'}  
        **🛠️ Equipment:** {", ".join(response_plan['equipment']) if response_plan['equipment'] else 'None'}  
        **📢 Instructions:** {response_plan['instructions']}
        """)

# Microphone recording function
def record_audio(duration=60):
    """Record audio for a fixed duration (seconds) with second-by-second display."""
    recognizer = sr.Recognizer()
    with sr.Microphone() as source:
        st.info(f"Recording for {duration} seconds... Speak clearly in Urdu.")

        frames = []  # Collect audio chunks
        recording_placeholder = st.empty()  # Placeholder to update seconds

        for i in range(duration):
            try:
                audio_chunk = recognizer.record(source, duration=1)  # Record 1 second
                frames.append(audio_chunk)
                recording_placeholder.success(f"Recording... {i+1} sec")
                time.sleep(0.5)  # small pause to let UI update nicely
            except Exception as e:
                st.error(f"Error during recording: {e}")
                return None

        st.success("Recording complete. Processing...")

        # Combine all audio chunks into one
        audio_data = sr.AudioData(b''.join([chunk.get_raw_data() for chunk in frames]),
                                  source.SAMPLE_RATE,
                                  source.SAMPLE_WIDTH)

        try:
            return recognizer.recognize_google(audio_data, language='ur')
        except sr.UnknownValueError:
            return "Could not understand the audio."
        except sr.RequestError:
            return "Could not request results from the service."

# Streamlit UI
st.set_page_config(page_title="1122 Emergency Detection", page_icon=":ambulance:", layout="wide")
st.title("🚨1122 Emergency Detection System")
st.subheader("AI-powered emergency call detection system")

st.markdown("""
This app processes Urdu audio calls to detect emergencies for Pakistan's 1122 emergency response system.
You can either upload an Urdu audio file or record using the microphone, and the system will analyze it to determine whether it indicates an emergency.
""")

# Sidebar option for selecting input type
input_type = st.sidebar.selectbox("Select Input Type", ["Microphone Input", "Audio File Upload"])

if input_type == "Microphone Input":
    st.markdown("Click below to start recording your voice in Urdu.")
    duration = st.slider("Select Recording Duration (seconds)", min_value=5, max_value=30, value=10, step=1)

    if st.button("🎙️ Start Recording"):
        urdu_text = record_audio(duration)

        if urdu_text:
            st.subheader("Audio Conversion")
            col1, col2 = st.columns(2)
            col1.text_area("Converted Urdu Text", urdu_text, height=150)

            translated_text = translator.translate(urdu_text, src='ur', dest='en').text if urdu_text else ""
            col2.text_area("Translated English Text", translated_text, height=150)

            if translated_text and "Error" not in translated_text:
                with st.spinner("Analyzing the text for emergencies..."):
                    prediction = predict_emergency(translated_text)
                    time.sleep(1)

                st.subheader("Prediction Result")
                if 'Normal_Call' in prediction:
                    st.markdown(
                        "<div style='padding: 20px; border-radius: 10px; background-color: green; color: white; font-size: 25px; width: 60%;'>"
                        "<strong>No Emergency:</strong> call is normal."
                        "</div>", unsafe_allow_html=True
                    )
                else:
                    st.markdown(
                        f"<div style='padding: 20px; border-radius: 10px; background-color: red; color: white; font-size: 25px; width: 60%;'>"
                        f"<strong>Emergency Detected:</strong> {prediction}"
                        f"</div>", unsafe_allow_html=True
                    )
                      # Show Emergency Response Plan
                show_emergency_response_plan(prediction)

elif input_type == "Audio File Upload":
    st.sidebar.title("Upload Urdu Audio File")
    audio_file = st.sidebar.file_uploader("Choose an Urdu audio file", type=["wav", "mp3"])

    if audio_file is not None:
        with st.spinner("Converting Urdu audio to text..."):
            urdu_text = audio_to_urdu_text(audio_file)
            time.sleep(1)

        st.subheader("Audio Conversion")
        col1, col2 = st.columns(2)
        col1.text_area("Converted Urdu Text", urdu_text, height=150)

        translated_text = translator.translate(urdu_text, src='ur', dest='en').text if urdu_text else ""
        col2.text_area("Translated English Text", translated_text, height=150)

        if translated_text and "Error" not in translated_text:
            with st.spinner("Analyzing the text for emergencies..."):
                prediction = predict_emergency(translated_text)
                time.sleep(1)

            st.subheader("Prediction Result")
            if 'Normal_Call' in prediction:
                st.markdown(
                    "<div style='padding: 20px; border-radius: 10px; background-color: green; color: white; font-size: 25px; width: 60%;'>"
                    "<strong>No Emergency:</strong> call is normal."
                    "</div>", unsafe_allow_html=True
                )
            else:
                st.markdown(
                    f"<div style='padding: 20px; border-radius: 10px; background-color: red; color: white; font-size: 25px; width: 60%;'>"
                    f"<strong>Emergency Detected:</strong> {prediction}"
                    f"</div>", unsafe_allow_html=True
                )
                # Show Emergency Response Plan
            show_emergency_response_plan(prediction)
               

st.markdown("---")
st.markdown("Developed with ❤️ for Pakistan's emergency response system.")

Overwriting test.py


In [ ]:
!streamlit run test.py